In [1]:
import os, time, glob, ast, re, warnings
warnings.filterwarnings("ignore")
from contextlib import contextmanager
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import agentic_bench as ab

pd.set_option("display.width", 170); pd.set_option("display.max_columns", 60)
HEART_CSV   = os.path.expanduser("~/Downloads/archive/heart_disease_cleveland.csv")
NYC_PARQUET = os.path.expanduser("~/Downloads/yellow_tripdata_2024-01.parquet")
AMAZON_CSV  = "data/amazon_beauty_50k.csv"

ab.cv_gain_from_dropping.__defaults__ = ((0,),)   # same speed-up as the main run, so numbers stay comparable
os.makedirs("outputs", exist_ok=True)
TARGET_SPLITS = 10
USE_AG = True            # AutoGluon in sections A and B

ModuleNotFoundError: No module named 'agentic_bench'

In [ ]:
df_heart, t_heart = ab.load_heart(HEART_CSV)

def load_amazon_csv(path, n=5000, seed=42):
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    d = pd.read_csv(path).dropna(subset=["text", "title", "rating"])
    d = d[(d.text.str.len() > 0)].sample(n=min(n, len(d)), random_state=seed).reset_index(drop=True)
    sia = SentimentIntensityAnalyzer()
    out = pd.DataFrame({
        "text_length": d.text.str.len(), "title_length": d.title.astype(str).str.len(),
        "word_count": d.text.str.split().str.len(), "exclaim_count": d.text.str.count("!"),
        "upper_ratio": d.text.apply(lambda s: sum(c.isupper() for c in s) / max(len(s), 1)),
        "vader_text": d.text.apply(lambda s: sia.polarity_scores(s)["compound"]),
        "vader_title": d.title.astype(str).apply(lambda s: sia.polarity_scores(s)["compound"]),
        "helpful_vote": d.helpful_vote, "verified": d.verified.astype(int),
        "sentiment": (d.rating >= 4).astype(int)})
    return out, "sentiment"

df_nyc, t_nyc = ab.load_nyc(NYC_PARQUET, n=5000)
df_amz, t_amz = load_amazon_csv(AMAZON_CSV, n=5000)
DATA = {"Heart_Disease": (df_heart, t_heart), "NYC_Taxi": (df_nyc, t_nyc), "Amazon_Reviews": (df_amz, t_amz)}
for k, (d, t) in DATA.items():
    print(f"{k}: {d.shape}, positive rate {d[t].mean():.2f}, missing cells {int(d.isna().sum().sum())}")

In [ ]:
def run_one_seed(k, seed, tag, data=None, conditions=ab.DEFAULT_CONDITIONS, leak=False, use_ag=True):
    """One split, all systems. Saved to outputs/<tag>_<dataset>_<seed>.csv only when it finishes."""
    p = f"outputs/{tag}_{k}_{seed}.csv"
    if os.path.exists(p):
        return pd.read_csv(p)
    ab.run_experiment({k: data if data is not None else DATA[k]}, seeds=[seed], conditions=conditions,
                      use_autogluon=use_ag, leak=leak, out_csv=p + ".part", verbose=False)
    os.replace(p + ".part", p)
    return pd.read_csv(p)

def combined(k):
    """Main study rows + AutoGluon rows + every extra split, no duplicates."""
    main = pd.read_csv("outputs/full_all.csv"); main = main[(main.dataset == k) & (main.system != "autogluon")]
    ag = pd.read_csv("outputs/full_autogluon.csv"); ag = ag[ag.dataset == k]
    extra = [pd.read_csv(f) for f in sorted(glob.glob(f"outputs/more_{k}_*.csv"))]
    return pd.concat([main, ag] + extra, ignore_index=True).drop_duplicates(["dataset", "system", "seed", "label_noise"])

SYS3 = ["single_agent", "graph_reviewer_fixed", "autogluon"]
def spread(df, systems=SYS3, metric="mcc"):
    b = df[df.system.isin(systems)]
    display(b.groupby(["dataset", "label_noise", "system"])[metric].agg(["mean", "std", "count"]).round(3).unstack("system"))
    p = b.pivot_table(index=["dataset", "label_noise", "seed"], columns="system", values=metric).reset_index()
    rows = []
    for a_ in [s for s in systems if s != "single_agent"]:
        p["d"] = p[a_] - p["single_agent"]
        g = p.groupby(["dataset", "label_noise"])["d"].agg(["mean", "std", "count"]); g["se"] = g["std"] / np.sqrt(g["count"])
        g["tie"] = g["mean"].abs() <= 2 * g["se"]; g.insert(0, "system", a_); rows.append(g.reset_index())
    out = pd.concat(rows); print(f"\n{metric}: each system minus single_agent, per seed (tie = within 2 standard errors)")
    display(out.round(3)); return out

In [ ]:
# keep splits already finished in the earlier partial Amazon run (24 rows = 4 noise levels x 6 systems)
old = "outputs/more_Amazon_Reviews.csv"
if os.path.exists(old):
    d = pd.read_csv(old)
    for seed, g in d.groupby("seed"):
        p = f"outputs/more_Amazon_Reviews_{seed}.csv"
        if len(g) == 24 and not os.path.exists(p):
            g.to_csv(p, index=False); print("kept finished Amazon split", seed)

for k in ["Amazon_Reviews", "NYC_Taxi"]:
    for seed in range(3, TARGET_SPLITS):
        t0 = time.time(); done = os.path.exists(f"outputs/more_{k}_{seed}.csv")
        run_one_seed(k, seed, "more", use_ag=USE_AG)
        print(k, "split", seed, "already done" if done else f"done in {(time.time()-t0)/60:.1f} min")
print({k: combined(k).seed.nunique() for k in ["Heart_Disease", "Amazon_Reviews", "NYC_Taxi"]})

In [ ]:
df_pay, t_pay = ab.load_nyc(NYC_PARQUET, n=5000, keep_payment_type=True)
raw = pd.read_parquet(NYC_PARQUET, columns=["total_amount"])
df_tot, t_tot = ab.load_nyc(NYC_PARQUET, n=5000)
df_tot = df_tot.copy(); df_tot["total_amount"] = raw.loc[df_tot.index, "total_amount"].values
NAT = {"payment_type": (df_pay, t_pay, "natpay"), "total_amount": (df_tot, t_tot, "nattot")}

for name, (d, t, tag) in NAT.items():
    for seed in range(TARGET_SPLITS):
        t0 = time.time(); done = os.path.exists(f"outputs/{tag}_NYC_Taxi_{seed}.csv")
        run_one_seed("NYC_Taxi", seed, tag, data=(d, t), conditions=[(0.0, 0.0)], use_ag=USE_AG)
        print(name, "split", seed, "already done" if done else f"done in {(time.time()-t0)/60:.1f} min")

res_nat = {}
for name, (d, t, tag) in NAT.items():
    r = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(f"outputs/{tag}_NYC_Taxi_*.csv"))], ignore_index=True)
    res_nat[name] = r
    print("\n=====", name, "kept as a feature,", r.seed.nunique(), "splits =====")
    display(r.groupby("system")["mcc"].agg(["mean", "std"]).round(3))
    fx = r[r.system == "graph_reviewer_fixed"]
    caught = fx.features_dropped.fillna("").str.split(",").apply(lambda L: name in L)
    print(f"fixed Reviewer dropped {name} in {int(caught.sum())} of {len(fx)} splits; MCC when it did: {fx[caught].mcc.mean():.3f}, when it did not: {fx[~caught].mcc.mean() if (~caught).any() else float('nan'):.3f}")
    v1 = r[r.system == "graph_reviewer_v1"]
    print(f"first (v1) Reviewer dropped it in {int(v1.features_dropped.fillna('').str.split(',').apply(lambda L: name in L).sum())} of {len(v1)} splits")

In [ ]:
def run_autogluon_preset(train, test, target, tag, preset, time_limit):
    from autogluon.tabular import TabularPredictor
    tr, te = train.drop(columns=[ab.RID]), test.drop(columns=[ab.RID])
    p = TabularPredictor(label=target, eval_metric="balanced_accuracy", verbosity=0,
                         path=f"/tmp/ag_{tag}_{int(time.time())}").fit(tr, time_limit=time_limit, presets=preset)
    prob = p.predict_proba(te.drop(columns=[target])).iloc[:, 1].values
    return {"metrics": ab.score(te[target], (prob >= 0.5).astype(int), prob), "model": p.model_best}

BQ_SEEDS = {"Heart_Disease": range(5), "NYC_Taxi": range(3), "Amazon_Reviews": range(3)}
BQ_TIME = 180
path = "outputs/ag_best_quality.csv"
bq = pd.read_csv(path) if os.path.exists(path) else pd.DataFrame(columns=["dataset", "seed", "label_noise"])
for k, seeds in BQ_SEEDS.items():
    df, target = DATA[k]; df = ab.add_rid(df)
    for seed in seeds:
        train, test = ab.make_split(df, target, seed)
        for ln, ir in [(0.0, 0.0), (0.30, 0.05)]:
            if ((bq.dataset == k) & (bq.seed == seed) & (bq.label_noise == ln)).any():
                continue
            tr_c, _ = ab.corrupt_train(train, target, k, ln, ir, seed)
            t0 = time.time()
            r = run_autogluon_preset(tr_c, test, target, f"bq_{k}_{seed}", "best_quality", BQ_TIME)
            bq = pd.concat([bq, pd.DataFrame([{"dataset": k, "seed": seed, "label_noise": ln, "system": "autogluon_best_quality",
                                               **r["metrics"], "model": r["model"], "seconds": round(time.time() - t0, 1)}])], ignore_index=True)
            bq.to_csv(path, index=False); print(k, seed, ln, round(r["metrics"]["mcc"], 3), f"{time.time()-t0:.0f}s")

med = pd.concat([combined(k) for k in BQ_SEEDS])
med = med[(med.system == "autogluon") & (med.label_noise.isin([0.0, 0.30]))]
keep = pd.concat([med.merge(bq[["dataset", "seed", "label_noise"]].drop_duplicates(), on=["dataset", "seed", "label_noise"]), bq])
single = pd.concat([combined(k) for k in BQ_SEEDS]); single = single[(single.system == "single_agent") & (single.label_noise.isin([0.0, 0.30]))]
single = single.merge(bq[["dataset", "seed", "label_noise"]].drop_duplicates(), on=["dataset", "seed", "label_noise"])
cmp_ = pd.concat([keep, single]).groupby(["dataset", "label_noise", "system"])["mcc"].agg(["mean", "std", "count"]).round(3).unstack("system")
display(cmp_)

In [ ]:
class LoggingLLM:
    def __init__(self, base): self.base, self.log = base, []
    def invoke(self, prompt):
        ans = self.base.invoke(prompt)
        self.log.append({"prompt": prompt, "answer": ans}); return ans

def analyse_strategy_log(log):
    rows = []
    for e in log:
        if "Pick ONE strategy" not in e["prompt"]:
            continue
        untried = ast.literal_eval(re.search(r"from (\[.*?\])\.", e["prompt"]).group(1))
        ans = e["answer"].strip().lower()
        named = next((s for s in untried if s in ans), None)
        rows.append({"untried": untried, "raw_answer": e["answer"].strip(), "named": named,
                     "valid": named is not None, "differs_from_rules": named is not None and named != untried[0]})
    return pd.DataFrame(rows)

LLM_DATASET = "NYC_Taxi"
LLM_SEEDS = range(TARGET_SPLITS)
path_rows, path_log = "outputs/llm_strategy_rows.csv", "outputs/llm_strategy_raw_answers.csv"
if not (os.path.exists(path_rows) and os.path.exists(path_log)):
    from langchain_ollama import OllamaLLM
    rows, logs = [], []
    df, t = DATA[LLM_DATASET]; df = ab.add_rid(df)
    for seed in LLM_SEEDS:
        train, test = ab.make_split(df, t, seed)
        for ln, ir in [(0.0, 0.0), (0.15, 0.05)]:
            tr_c, _ = ab.corrupt_train(train, t, LLM_DATASET, ln, ir, seed)
            llm_log = LoggingLLM(OllamaLLM(model="llama3.2"))
            app = ab.build_graph(use_reviewer=True, llm=llm_log, reviewer_mode="fixed", strategy_mode="llm")
            o = ab.run_graph(app, tr_c, test, t, LLM_DATASET)
            a = analyse_strategy_log(llm_log.log)
            rows.append({"seed": seed, "label_noise": ln, "mcc": o["metrics"]["mcc"], "balanced_acc": o["metrics"]["balanced_acc"],
                         "strategy_prompts": len(a), "valid_answers": int(a.valid.sum()) if len(a) else 0,
                         "different_from_rules": int(a.differs_from_rules.sum()) if len(a) else 0,
                         "strategies": " > ".join(h["strategy"] for h in o["history"])})
            logs += [{"seed": seed, "label_noise": ln, **r} for r in a.to_dict("records")]
            print(rows[-1])
    pd.DataFrame(rows).to_csv(path_rows, index=False); pd.DataFrame(logs).to_csv(path_log, index=False)
llm_rows, llm_raw = pd.read_csv(path_rows), pd.read_csv(path_log)
print("strategy prompts:", len(llm_raw), "| valid answers:", int(llm_raw.valid.sum()), "| different from the rules' pick:", int(llm_raw.differs_from_rules.sum()))
display(llm_raw.raw_answer.str.lower().value_counts().head(10))
rules = combined(LLM_DATASET); rules = rules[(rules.system == "graph_reviewer_fixed") & (rules.label_noise.isin([0.0, 0.15]))]
m = llm_rows.merge(rules[["seed", "label_noise", "mcc"]], on=["seed", "label_noise"], suffixes=("_llm", "_rules"))
print("MCC, rules versus language model, same splits (mean):", round(m.mcc_rules.mean(), 3), round(m.mcc_llm.mean(), 3), "| identical in", int((m.mcc_rules.round(6) == m.mcc_llm.round(6)).sum()), "of", len(m))

In [ ]:
VARIANTS = {
    "baseline": {},
    "strict_label_gate": {"min_gain": 0.10, "min_win": 0.90},
    "loose_label_gate": {"min_gain": 0.05, "min_win": 0.50},
    "shifted_cutoffs": {"label_cutoffs": (0.05, 0.10, 0.20)},
    "strict_leak_checks": {"_leak": (0.98, 0.90, 0.40)},   # (single-feature AUC, score floor, score drop)
    "loose_leak_checks": {"_leak": (0.90, 0.80, 0.20)},
}
SENS_SEEDS = range(10)
SENS_CONDS = [(0.0, 0.0), (0.15, 0.05), (0.30, 0.05)]

@contextmanager
def leak_thresholds(auc, floor, drop):
    a, b = ab.leakage_suspects.__defaults__, ab.score_jump_leaks.__defaults__
    ab.leakage_suspects.__defaults__, ab.score_jump_leaks.__defaults__ = (auc,), (floor, drop)
    try: yield
    finally: ab.leakage_suspects.__defaults__, ab.score_jump_leaks.__defaults__ = a, b

def sens_run(kw, seed, cond, leak):
    kw = dict(kw); lk = kw.pop("_leak", None)
    df, t = DATA["Heart_Disease"]
    d = ab.add_rid(ab.add_leaky_feature(df, t) if leak else df)
    train, test = ab.make_split(d, t, seed)
    tr_c, info = ab.corrupt_train(train, t, "Heart_Disease", cond[0], cond[1], seed)
    app = ab.build_graph(use_reviewer=True, llm=None, reviewer_mode="fixed", **kw)
    if lk:
        with leak_thresholds(*lk): o = ab.run_graph(app, tr_c, test, t, "Heart_Disease")
    else:
        o = ab.run_graph(app, tr_c, test, t, "Heart_Disease")
    single = ab.run_single_agent(tr_c, test, t)["metrics"]["mcc"]
    return {"mcc": o["metrics"]["mcc"], "single_mcc": single, "n_dropped_label": len(o["dropped_label"]),
            "features_dropped": ",".join(o.get("drop_cols", [])),
            "leak_flagged": "leaky_feature" in o.get("drop_cols", [])}

path = "outputs/sensitivity_heart.csv"
sens = pd.read_csv(path) if os.path.exists(path) else pd.DataFrame(columns=["variant", "seed", "label_noise", "leak"])
def have(v, s, c, l): return ((sens.variant == v) & (sens.seed == s) & (sens.label_noise == c[0]) & (sens.leak == l)).any()
for v, kw in VARIANTS.items():
    t0 = time.time()
    for seed in SENS_SEEDS:
        for cond, leak in [(c, False) for c in SENS_CONDS] + [((0.0, 0.0), True)]:
            if have(v, seed, cond, leak): continue
            r = sens_run(kw, seed, cond, leak)
            sens = pd.concat([sens, pd.DataFrame([{"variant": v, "seed": seed, "label_noise": cond[0], "leak": leak, **r}])], ignore_index=True)
        sens.to_csv(path, index=False)
    print(v, "done", round((time.time() - t0) / 60, 1), "min")

clean = sens[~sens.leak.astype(bool)].copy(); clean["diff_vs_single"] = clean.mcc - clean.single_mcc
tab = clean.groupby(["variant", "label_noise"])["diff_vs_single"].mean().round(3).unstack("label_noise")
tab["labels_removed (mean)"] = clean.groupby("variant")["n_dropped_label"].mean().round(1)
tab["false leak alarms (clean data)"] = clean[clean.label_noise == 0.0].groupby("variant").features_dropped.apply(lambda s: int((s.fillna("") != "").sum()))
tab["planted leak caught (of 10)"] = sens[sens.leak.astype(bool)].groupby("variant").leak_flagged.sum()
print("Fixed Reviewer minus single pass, MCC on Heart Disease, for each setting")
display(tab)

In [ ]:
final = pd.concat([combined(k) for k in ["Heart_Disease", "Amazon_Reviews", "NYC_Taxi"]], ignore_index=True)
print(final.groupby("dataset").seed.nunique().to_dict(), "splits per dataset")
sp = spread(final)
sp.to_csv("outputs/final_spread.csv", index=False)
mean_tab = final[final.system.isin(SYS3)].groupby(["dataset", "label_noise", "system"])["mcc"].mean().round(3).unstack("system")
mean_tab.to_csv("outputs/final_mcc_table.csv"); display(mean_tab)

print("\nTaxi, clean data: recall and specificity (the 'says tipped for everyone' check)")
display(final[(final.dataset == "NYC_Taxi") & (final.label_noise == 0.0)].groupby("system")[["balanced_acc", "mcc", "f1", "recall", "specificity"]].mean().round(3))

print("\nFalse leak alarms: how often the fixed Reviewer dropped a feature on clean data with no leak")
fx = final[(final.system == "graph_reviewer_fixed") & (final.label_noise == 0.0)]
display(fx.groupby("dataset").apply(lambda g: pd.Series({"runs": len(g), "runs_with_a_dropped_feature": int((g.n_features_dropped > 0).sum())})))

print("\nBest_quality and natural-leak results are in the cells above; saved files:")
print(sorted(glob.glob("outputs/final_*.csv")) + ["outputs/ag_best_quality.csv", "outputs/sensitivity_heart.csv", "outputs/llm_strategy_raw_answers.csv"])